# Part 2: Modern Memory Patterns — LCEL 체인에 메모리 추가하기

## Part 1 vs Part 2: 무엇이 달라졌나요?

| 항목 | Part 1 (Legacy, 01~07번) | Part 2 (Modern, 08~14번) |
|------|--------------------------|--------------------------|
| 메모리 클래스 | `ConversationBufferMemory` 등 전용 클래스 | `ChatMessageHistory` + `RunnableWithMessageHistory` |
| 체인 방식 | `ConversationChain` (deprecated) | LCEL 파이프라인 (`\|` 연산자) |
| 저장 방식 | 인메모리 전용 | 인메모리 · SQLite · PostgreSQL 등 |
| 세션 분리 | 제한적 | `session_id` 기반으로 자유롭게 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 공식 권장 방식 |

---

## 이번 노트북 학습 목표

- `RunnablePassthrough.assign()`으로 메모리 변수를 LCEL 체인에 주입하는 흐름을 이해해요.
- `memory.save_context()`를 직접 호출해 대화를 저장하는 **수동 관리 패턴**을 익혀요.
- 수동 관리의 불편함을 느끼고, 다음 노트북(10번)의 **자동 관리 패턴**으로 나아갈 준비를 해요.

## 사전 지식

- LCEL 파이프라인 기본 (`\|` 연산자, `Runnable` 인터페이스)
- `ChatPromptTemplate`과 `MessagesPlaceholder` 사용법
- `ConversationBufferMemory`의 `memory_key` / `return_messages` 옵션

> 이 노트북은 **구식 메모리 클래스를 LCEL과 수동으로 연결하는 과도기적 패턴**을 다뤄요. 실무에서는 10번 이후의 `RunnableWithMessageHistory` 방식을 우선 사용하세요.


In [ ]:
from dotenv import load_dotenv
from operator import itemgetter
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

## 1. 기본 설정

LCEL 체인과 메모리를 설정해요. 여기서 핵심은 `MessagesPlaceholder`입니다. 이 자리표시자(placeholder)에 메모리에서 불러온 대화 이력이 자동으로 채워져요.


### LCEL + 메모리 통합 흐름

```mermaid
flowchart LR
    ENV[환경 변수 로드] --> MEM[ConversationBufferMemory<br/>memory_key=chat_history]
    ENV --> MODEL[ChatOpenAI]
    ENV --> PROMPT[ChatPromptTemplate<br/>MessagesPlaceholder 포함]
    MEM --> ASSIGN[RunnablePassthrough.assign<br/>chat_history 주입]
    ASSIGN --> PROMPT
    PROMPT --> MODEL
    MODEL --> PARSER[StrOutputParser]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class ENV input
    class MEM storage
    class ASSIGN,PROMPT,MODEL process
    class PARSER output
```

> `return_messages=True` 설정이 중요해요. `ChatPromptTemplate`은 문자열이 아니라 메시지 객체 리스트를 받아야 하기 때문이에요.


In [ ]:
# ---------------------------------------------------
# 모델, 프롬프트, 메모리 기본 설정
# ---------------------------------------------------

# 1단계: LLM 모델 초기화
# - temperature 기본값: 0이 아닌 기본값 사용 (창의적 응답 허용)
model = ChatOpenAI(model="gpt-4o-mini")

# 2단계: 대화형 프롬프트 생성
# - MessagesPlaceholder: 메모리에서 꺼낸 대화 이력이 채워지는 자리표시자
# - variable_name은 memory의 memory_key와 반드시 일치해야 함
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful chatbot"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 3단계: 메모리 생성
# - return_messages=True: 문자열이 아닌 메시지 객체 리스트로 반환 (ChatPromptTemplate 호환)
# - memory_key="chat_history": MessagesPlaceholder의 variable_name과 동일하게 설정
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

## 2. 메모리를 체인에 통합

`RunnablePassthrough.assign()`을 사용하면 입력 딕셔너리에 새 키를 추가할 수 있어요. 여기서는 메모리에서 꺼낸 `chat_history`를 체인에 자동으로 끼워 넣어요.

### 수동 주입 흐름: RunnablePassthrough + itemgetter

```mermaid
flowchart LR
    LOAD[memory.load_memory_variables] --> LAMBDA[RunnableLambda]
    LAMBDA --> GET[itemgetter<br/>chat_history 키 추출]
    GET --> ASSIGN[RunnablePassthrough.assign<br/>chat_history=...]
    ASSIGN --> PROMPT --> MODEL --> PARSER[StrOutputParser]

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class LOAD storage
    class LAMBDA,GET,ASSIGN,PROMPT,MODEL process
    class PARSER output
```

> `itemgetter("chat_history")`는 `memory_key`와 동일한 이름을 사용해야 해요. 이름이 다르면 `KeyError`가 발생합니다.


In [ ]:
# ============================================================
# TODO: RunnablePassthrough.assign()으로 체인을 구성하세요
# 힌트: memory.load_memory_variables를 RunnableLambda로 감싸고,
#       itemgetter("chat_history")로 메모리 키를 추출한 뒤,
#       prompt | model | StrOutputParser()로 연결하세요
# 예상 결과: chain.invoke({"input": "..."})로 대화가 가능해야 합니다
# ============================================================

# ---------------------------------------------------
# 체인 구성: 메모리 → 프롬프트 → 모델 → 출력 파서
# ---------------------------------------------------

# 1단계: RunnablePassthrough.assign()으로 chat_history 변수를 자동 주입
# - memory.load_memory_variables: 메모리에서 대화 이력을 불러오는 함수
# - RunnableLambda: 일반 함수를 Runnable로 변환
# - itemgetter("chat_history"): 딕셔너리에서 chat_history 키만 추출


## 3. 체인 실행 및 메모리 저장

체인을 실행한 후 응답을 메모리에 저장해요. 이 **수동 저장** 단계가 이 패턴의 핵심 불편함이에요. 저장을 빠뜨리면 다음 대화에서 이전 내용을 기억하지 못해요.

### 실행 → 저장 흐름

```mermaid
flowchart TD
    INPUT[사용자 입력] --> INVOKE[chain.invoke]
    INVOKE --> RESPONSE[AI 응답]
    RESPONSE --> PRINT[출력]
    INPUT --> SAVE[memory.save_context<br/>inputs + outputs]
    RESPONSE --> SAVE
    SAVE --> UPDATED[chat_history 업데이트]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class INPUT input
    class INVOKE,SAVE process
    class RESPONSE,PRINT output
    class UPDATED storage
```

> `save_context()`를 호출하지 않으면 메모리가 갱신되지 않아요. 다음 노트북(10번)의 `RunnableWithMessageHistory`는 이 단계를 자동으로 처리해 줍니다.


In [ ]:
# ============================================================
# TODO: 첫 번째 대화를 실행하고 메모리에 저장하세요
# 힌트: chain.invoke()로 응답을 받은 뒤, memory.save_context()로 대화를 저장하세요
# 예상 결과: AI가 인사를 하고, 메모리에 대화가 저장되어야 합니다
# ============================================================

# ---------------------------------------------------
# 첫 번째 대화 실행 및 메모리 수동 저장
# ---------------------------------------------------

# 1단계: 첫 번째 질문 실행 (자기소개)


# 2단계: 응답을 메모리에 수동 저장
# - save_context(): inputs에 사용자 입력, outputs에 AI 응답을 넣어야 함
# - 이 단계를 빠뜨리면 다음 대화에서 이전 내용을 기억하지 못함


## 4. 이전 대화 기억 확인

이전 대화를 기억하고 있는지 확인해봅시다.


In [ ]:
# ============================================================
# TODO: 두 번째 대화를 실행하고 메모리에 저장하세요
# 힌트: chain.invoke()로 응답을 받은 뒤, memory.save_context()로 대화를 저장하세요
# 예상 결과: AI가 이전에 알려준 이름을 기억하여 답변해야 합니다
# ============================================================

# ---------------------------------------------------
# 두 번째 대화 실행 및 메모리 수동 저장
# ---------------------------------------------------

# 1단계: 두 번째 질문 실행 (이름을 기억하는지 확인)


# 2단계: 응답을 메모리에 수동 저장


## 5. 커스텀 ConversationChain 구현

`save_context()` 호출을 매번 수동으로 하기가 번거롭죠? `Runnable`을 상속해 메모리 저장을 `invoke()` 안에서 자동으로 처리하는 커스텀 클래스를 만들어봐요.

### MyConversationChain 내부 구조

```mermaid
flowchart TD
    QUERY[query 문자열] --> INVOKE[MyConversationChain.invoke]
    INVOKE --> CHAIN_EXEC[self.chain.invoke]
    CHAIN_EXEC --> ANSWER[AI 응답]
    QUERY --> AUTO_SAVE[memory.save_context<br/>자동 호출]
    ANSWER --> AUTO_SAVE
    AUTO_SAVE --> HISTORY[chat_history 갱신]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class QUERY input
    class INVOKE,CHAIN_EXEC,AUTO_SAVE process
    class ANSWER output
    class HISTORY storage
```


In [ ]:
# ============================================================
# TODO: MyConversationChain 클래스를 구현하세요
# 힌트: Runnable을 상속받아 invoke() 메서드에서 체인 실행 + 메모리 저장을 자동화하세요
# 예상 결과: conversation_chain.invoke("질문")만으로 대화 + 메모리 저장이 한 번에 되어야 합니다
# ============================================================

# ---------------------------------------------------
# 커스텀 ConversationChain 클래스 정의
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# MyConversationChain 인스턴스 생성 및 테스트
# ---------------------------------------------------

# 1단계: 새 LLM, 프롬프트, 메모리 생성 (이전 대화와 분리)
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful chatbot"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

# 2단계: MyConversationChain 인스턴스 생성


In [ ]:
# ---------------------------------------------------
# 테스트: 첫 번째 대화 (자기소개)
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# 테스트: 이름 기억 확인
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# 테스트: 언어 변경 요청
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# 테스트: 영어 답변 확인
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# 테스트: 전체 대화 기록 확인
# ---------------------------------------------------


## 핵심 요약

이 노트북에서 배운 내용을 정리해요.

### 수동 메모리 관리 패턴 핵심 구조

| 단계 | 코드 | 역할 |
|------|------|------|
| 메모리 생성 | `ConversationBufferMemory(return_messages=True)` | 대화 이력 저장소 |
| 체인에 주입 | `RunnablePassthrough.assign(chat_history=...)` | 메모리 → 프롬프트 연결 |
| 체인 실행 | `chain.invoke({"input": "..."})` | LLM 호출 |
| 저장 (수동) | `memory.save_context(inputs, outputs)` | 대화 기록 갱신 |

### 주의사항

- `save_context()` 호출을 빠뜨리면 다음 턴에서 이전 대화를 기억하지 못해요.
- `memory_key`와 `MessagesPlaceholder`의 `variable_name`은 반드시 일치해야 해요.
- `return_messages=True`를 빠뜨리면 `ChatPromptTemplate`이 메시지 객체를 받지 못해 오류가 발생해요.

### 다음 단계 예고

**09번**: SQLite로 대화 이력을 영구 저장하는 방법을 배워요. 앱을 재시작해도 대화 기록이 사라지지 않아요.

**10번**: `RunnableWithMessageHistory`로 `save_context()` 호출을 없애는 **자동 메모리 관리 패턴**을 익혀요. 이것이 LangChain 1.0.x의 공식 권장 방식이에요.
